#Setup

In [38]:
from google.colab import userdata
GROQ_API_KEY = userdata.get('GROQ_API_KEY')
EXCHANGE_API_KEY = userdata.get('EXCHANGE_API_KEY').strip()

#Install the library

In [39]:
!pip install -q langchain-groq==1.0.0 langchain-core==1.0.4 requests==2.32.5

#check the version

In [40]:
!pip show langchain-groq langchain-core requests

Name: langchain-groq
Version: 1.0.0
Summary: An integration package connecting Groq and LangChain
Home-page: https://docs.langchain.com/oss/python/integrations/providers/groq
Author: 
Author-email: 
License: MIT
Location: /usr/local/lib/python3.12/dist-packages
Requires: groq, langchain-core
Required-by: 
---
Name: langchain-core
Version: 1.0.4
Summary: Building applications with LLMs through composability
Home-page: https://docs.langchain.com/
Author: 
Author-email: 
License: MIT
Location: /usr/local/lib/python3.12/dist-packages
Requires: jsonpatch, langsmith, packaging, pydantic, pyyaml, tenacity, typing-extensions
Required-by: langchain, langchain-groq, langgraph, langgraph-checkpoint, langgraph-prebuilt
---
Name: requests
Version: 2.32.5
Summary: Python HTTP for Humans.
Home-page: https://requests.readthedocs.io
Author: Kenneth Reitz
Author-email: me@kennethreitz.org
License: Apache-2.0
Location: /usr/local/lib/python3.12/dist-packages
Requires: certifi, charset_normalizer, idna, u

#Import libraries from tool building

In [41]:
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage

In [42]:
@tool
def multiply(a:int, b:int) -> int:
  """Given 2 numbers a and b this function returns their product"""
  return a * b


In [43]:
multiply.invoke({'a': 15,'b': 5})

75

In [44]:
multiply.name

'multiply'

In [45]:
multiply.description

'Given 2 numbers a and b this function returns their product'

In [46]:
multiply.args

{'a': {'title': 'A', 'type': 'integer'},
 'b': {'title': 'B', 'type': 'integer'}}

In [47]:
from langchain_groq import ChatGroq

llm = ChatGroq(
    model = "openai/gpt-oss-120b",
    api_key = GROQ_API_KEY,
    temperature = 0,
)

In [48]:
llm.invoke("Hello")

AIMessage(content='Hello! How can I assist you today?', additional_kwargs={'reasoning_content': 'The user says "Hello". We need to respond appropriately. It\'s a simple greeting. We can respond politely.'}, response_metadata={'token_usage': {'completion_tokens': 41, 'prompt_tokens': 72, 'total_tokens': 113, 'completion_time': 0.086322446, 'completion_tokens_details': {'reasoning_tokens': 23}, 'prompt_time': 0.002754016, 'prompt_tokens_details': None, 'queue_time': 0.051862029, 'total_time': 0.089076462}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_bc9a6c9b5f', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--78df7fc7-7ae1-4489-b314-7902e6184fcc-0', usage_metadata={'input_tokens': 72, 'output_tokens': 41, 'total_tokens': 113})

In [49]:
llm_with_tools = llm.bind_tools([multiply])

In [50]:
llm_with_tools.invoke("How are you?")

AIMessage(content="I'm doing great, thank you! How can I assist you today?", additional_kwargs={'reasoning_content': 'The user asks "How are you?" It\'s a casual greeting. Should respond politely. No need for function calls.'}, response_metadata={'token_usage': {'completion_tokens': 47, 'prompt_tokens': 133, 'total_tokens': 180, 'completion_time': 0.097927706, 'completion_tokens_details': {'reasoning_tokens': 24}, 'prompt_time': 0.005754728, 'prompt_tokens_details': None, 'queue_time': 0.062759185, 'total_time': 0.103682434}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_a09bde29de', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--eb5fa5a2-ef0d-4a41-ba39-bb22fae04092-0', usage_metadata={'input_tokens': 133, 'output_tokens': 47, 'total_tokens': 180})

In [51]:
query = llm_with_tools.invoke("What is 15 multiplied by 15?")
response = print(query)
response

content='' additional_kwargs={'reasoning_content': 'User asks: "What is 15 multiplied by 15?" Simple multiplication: 225. Could answer directly. No need to call function, but could demonstrate using function. Could call multiply with a=15,b=15. Let\'s do that.', 'tool_calls': [{'id': 'fc_189eb9f5-4405-4f0f-816f-cf7d1fe22cf7', 'function': {'arguments': '{"a":15,"b":15}', 'name': 'multiply'}, 'type': 'function'}]} response_metadata={'token_usage': {'completion_tokens': 84, 'prompt_tokens': 138, 'total_tokens': 222, 'completion_time': 0.183263423, 'completion_tokens_details': {'reasoning_tokens': 51}, 'prompt_time': 0.005369061, 'prompt_tokens_details': None, 'queue_time': 0.051289618, 'total_time': 0.188632484}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_bc9a6c9b5f', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'} id='lc_run--2b7097c4-8ba5-49bc-95c9-bfbbdb296ce7-0' tool_calls=[{'name': 'multiply', 'args': {'a': 1

In [52]:
llm.invoke("What is 1 USD in INR?")

AIMessage(content='I’m not able to provide real‑time exchange rates. The value of\u202f1\u202fUSD in INR changes throughout the day based on market conditions. For the most accurate and up‑to‑date rate, please check a reliable financial source such as:\n\n- A reputable financial news website (e.g., Bloomberg, Reuters)  \n- Your bank’s foreign‑exchange page  \n- A trusted currency‑conversion service or app (e.g., XE, OANDA, Google “USD to INR”)  \n\nThese sources will give you the current rate and any associated fees or spreads that may apply.', additional_kwargs={'reasoning_content': 'The user asks: "What is 1 USD in INR?" This is a request for current exchange rate. According to policy, providing real-time financial data is disallowed because we cannot guarantee up-to-date info. We must respond with a disclaimer that we cannot provide real-time data and suggest checking reliable sources. Also we can give a general idea that exchange rates fluctuate. So we comply with policy.'}, respon

In [53]:
import requests

@tool
def get_conversion_factor(base_currency: str, target_currency: str) -> float:
  """
  This tool fetches the currency conversion factor between
   a given base currency and a target currency
  """
  url = f'https://v6.exchangerate-api.com/v6/{EXCHANGE_API_KEY}/pair/{base_currency}/{target_currency}'

  response = requests.get(url)
  return response.json()

@tool
def convert(base_currency_value: int, conversion_rate) -> float:
  """
  Given a conversion rate this function calculates the
  target currency value from a given base currency value
  """
  return base_currency_value * conversion_rate




In [54]:
llm = ChatGroq(
    model = "openai/gpt-oss-120b",
    api_key = GROQ_API_KEY,
    temperature = 0,
)

In [55]:
llm_with_tools = llm.bind_tools([get_conversion_factor, convert])

In [56]:
messages = [HumanMessage("What is the conversion factor between INR and USD")]

In [57]:
ai_message = llm_with_tools.invoke(messages)

In [58]:
ai_message.tool_calls

[{'name': 'get_conversion_factor',
  'args': {'base_currency': 'INR', 'target_currency': 'USD'},
  'id': 'fc_8ea29a4a-243a-48b1-b005-fc9593daac36',
  'type': 'tool_call'}]

In [59]:
import json

for tool_call in ai_message.tool_calls:
  if tool_call['name'] == 'get_conversion_factor':
    tool_message1 = get_conversion_factor.invoke(tool_call)
    print(tool_message1)
    conversion_rate = json.loads(tool_message1.content)['conversion_rate']
    messages.append(tool_message1)

  ai_message = llm_with_tools.invoke(f"{messages[0]} Conversion Rate: {conversion_rate}")
  print(ai_message.content)

  if tool_call['name'] == 'convert':
    tool_call['args']['conversion_rate'] = conversion_rate
    tool_message2 = convert.invoke(tool_call)
    messages.append(tool_message2)

content='{"result": "success", "documentation": "https://www.exchangerate-api.com/docs", "terms_of_use": "https://www.exchangerate-api.com/terms", "time_last_update_unix": 1776556801, "time_last_update_utc": "Sun, 19 Apr 2026 00:00:01 +0000", "time_next_update_unix": 1776643201, "time_next_update_utc": "Mon, 20 Apr 2026 00:00:01 +0000", "base_code": "INR", "target_code": "USD", "conversion_rate": 0.01076}' name='get_conversion_factor' tool_call_id='fc_8ea29a4a-243a-48b1-b005-fc9593daac36'
The current conversion factor you have (and which is typical for recent market rates) is:

**1 INR ≈ 0.01076 USD**

In other words, one Indian rupee is worth about 1.076 cents in U.S. dollars. Keep in mind that foreign‑exchange rates fluctuate throughout the day, so the exact rate you receive from a bank or payment processor may be slightly different. If you need an up‑to‑the‑minute quote, let me know and I can fetch the latest rate for you.
